# Colab Dry-Run: Real Production Training Path, 6 Steps (TIP-009c, Gate 0)

This notebook closes Gate 0. It replaces every pending Colab plan before it.
Its only job is to run the **actual production training entry point**
(`accelerate launch ... starVLA/training/train_starvla.py`), not a
lighter substitute, for exactly 6 steps, and capture the evidence Gate 0
needs: both dataset splits load through `build_dataloader` with no
`ValueError`, the model builds and selectively reloads the 6 checkpoint
modules from D22's 3 learning-rate groups, `wm_loss` and `action_loss` are
both non-zero and finite across all 6 steps (proving the world model
actually participates, not just the action head), one checkpoint gets
written with a measured size and wall time (for C30's future resume-path
fix), and peak VRAM stays under the ~49 GB floor a 3.08B-parameter full
finetune implies.

**Required GPU: A100 80GB. Nothing else.** L4, T4, and A100 40GB are not
enough -- AdamW fp32 states (~24.6 GB) + master weights (~12.3 GB) +
bf16 params/grads (~12.3 GB) already total ~49 GB before activations. S0
below checks this and halts the notebook immediately if the runtime is
wrong -- do not skip past that warning.

**A100 budget for this run: 30 minutes max.** This is the first pack
allowed to touch an A100 at all, and only because proving Gate 0 is what an
A100 run itself is for. Everything checkable without a GPU (whether
`delete_pause_frame=False` actually reaches the production dataloader,
whether `delete_pause_frame=True` still raises) was already proven on the
laptop -- see `ur10e/results/pause_frame_flag_proof.md` and
`ur10e/src/prove_pause_frame_flag.py`. This notebook does not repeat that
proof; it builds on it.

**Three things this notebook works around, none by editing an upstream
file:**

1. `starVLA/dataloader/gr00t_lerobot/datasets.py`'s steps cache keys on two
   hardcoded filenames regardless of `delete_pause_frame` (an `# @BUG`
   comment overrides the config-aware key it computes) -- S4 below clears
   both filenames before its own load check, same as the laptop proof
   script, so a stale cache from an earlier run can't mask a real failure.
2. `train_starvla.py` calls `get_logger(__name__)` and then `logger.info(...)`
   for its per-step loss line and LR-group summary, but never configures a
   logging handler and never sets `ACCELERATE_LOG_LEVEL` -- verified on the
   laptop that this makes every one of those lines vanish silently (Python's
   `logging.lastResort` handler filters at WARNING, and nothing in this
   codebase or accelerate attaches a real handler). Without a fix, this
   notebook could not recover per-step losses at all. S6 below drops a
   `sitecustomize.py` on `PYTHONPATH` that calls `logging.basicConfig` --
   Python auto-imports it at interpreter startup for every process that
   sees it on `PYTHONPATH`, so `train_starvla.py` itself is never touched.
3. **Found on the first real Colab run of this notebook (2026-08-21):**
   `normalize_dotlist_args` (`trainer_utils/trainer_tools.py`) only
   recognizes CLI overrides that start with `--`; anything else -- bare
   `key=value`, which is what every override in this pack's own S6 command
   used at first -- falls into an `else: pass  # skip orphaned values`
   branch and is silently dropped. All ten `key=value` overrides that first
   run passed were discarded without any warning, so the run built the
   model from `ur10e_ft.yaml`'s own defaults instead (including
   `/content/Qwen3-VL-2B-Instruct`, which doesn't exist -- only
   `/content/models/Qwen3-VL-2B-Instruct` does -- and crashed there).
   `scripts/run_vlajepa_libero_ft.sh`, the only other caller of this
   script in the repo, never exercises this path at all: it passes no
   overrides beyond `--config_yaml`. S6 below now prefixes every override
   with `--`; verified against `accelerate`'s own launch parser (which
   uses `argparse.REMAINDER` for the script args, so `--key=value` passes
   through to `train_starvla.py` untouched, not consumed by `accelerate
   launch` itself).

**One missing dependency, then one broken wheel, found across the second
and third real Colab runs (2026-08-21):** `QWen3.py` (and `QWen2_5.py`)
hardcode `attn_implementation="flash_attention_2"` in the
`from_pretrained(...)` call -- not read from `qwenvl.attn_implementation`
in the config despite that key existing -- and `flash-attn` is not in
`requirements.txt` at all. S2 below installs `ninja` (parallelizes a
from-source build) and `flash-attn --no-build-isolation`, then verifies
the install by actually importing `flash_attn_2_cuda` -- a prebuilt wheel
that doesn't match this Colab session's exact torch/CUDA/cxx11-ABI build
installs with exit code 0 but raises `undefined symbol` at import time,
which is what the third run hit. If the import check fails, S2
automatically uninstalls and reinstalls with
`FLASH_ATTENTION_FORCE_BUILD=TRUE` -- flash-attn's own documented escape
hatch to skip the prebuilt-wheel lookup and compile from source against
whatever torch is actually installed, guaranteeing ABI match.

**Design principle, unchanged from TIP-009's notebook (C24): every stage
below catches its own failures, records them, and lets the notebook keep
going.** The one deliberate exception is S0 -- a wrong GPU stops the
notebook outright, on purpose, because nothing after it is worth running.

Run all cells top to bottom. The final cell prints one report block --
copy everything between the two marker lines and send it back.

In [ ]:
import ast
import json
import os
import re
import subprocess
import sys
import threading
import time
import traceback
from pathlib import Path

REPO_DIR = "/content/VLA-JEPA"
CONFIG_PATH = f"{REPO_DIR}/ur10e/configs/ur10e_ft.yaml"
ENV_DIR = "/content/env-train"
ENV_PYTHON = f"{ENV_DIR}/bin/python"
ENV_BIN = f"{ENV_DIR}/bin"
HF_USER = "DuyBao44DOCer"  # Hugging Face username -- different from the GitHub username DuyBaoDOCer

REPORT = {
    "s0_gpu": "NOT RUN",
    "s0_vram_gb": "NOT RUN",
    "s1_commit": "NOT RUN",
    "s1_status": "NOT RUN",
    "s2_status": "NOT RUN",
    "s2_deepspeed_version": "NOT RUN",
    "s2_deepspeed_install_s": "NOT RUN",
    "s3_status": "NOT RUN",
    "s4_status": "NOT RUN",
    "s4_train_total_steps": "NOT RUN",
    "s4_heldout_total_steps": "NOT RUN",
    "s4_train_trajectories": "NOT RUN",
    "s4_heldout_trajectories": "NOT RUN",
    "s5_status": "NOT RUN",
    "s6_status": "NOT RUN",
    "s6_returncode": "NOT RUN",
    "s7_status": "NOT RUN",
}
TRACEBACKS = {}
STAGE_STATUS = {}  # "S0".."S7" -> "OK" / "FAILED"

print("Report state initialized. Fields fill in as sections below run.")

## S0) GATE: confirm A100 80GB, or stop here

No try/except on this cell, on purpose -- everything after it assumes a
real A100 80GB, and continuing on the wrong GPU would just burn Colab time
producing numbers that answer nothing. A failure here raises and halts
"Run all" immediately.

In [ ]:
gpu_query = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
    capture_output=True, text=True,
)
gpu_line = gpu_query.stdout.strip().splitlines()[0] if gpu_query.stdout.strip() else ""
print("nvidia-smi output:", gpu_line)

if gpu_query.returncode != 0 or not gpu_line:
    STAGE_STATUS["S0"] = "FAILED"
    raise RuntimeError(
        "=" * 70 + "\n"
        "nvidia-smi failed or returned nothing -- no GPU attached to this runtime.\n"
        "Switch Runtime type to a GPU runtime (A100) and re-run.\n"
        + "=" * 70
    )

gpu_name, gpu_mem_mb_str = [p.strip() for p in gpu_line.split(",")]
gpu_mem_mb = float(gpu_mem_mb_str)
gpu_mem_gb = gpu_mem_mb / 1024

REPORT["s0_gpu"] = gpu_name
REPORT["s0_vram_gb"] = f"{gpu_mem_gb:.1f}"

is_a100 = "A100" in gpu_name.upper()
# A100 80GB reports ~80994-81920 MiB depending on driver; A100 40GB reports
# ~40536-40960 MiB. 70000 MiB safely separates the two variants.
is_80gb = gpu_mem_mb >= 70000

print(f"GPU: {gpu_name}, VRAM: {gpu_mem_gb:.1f} GB")

if not (is_a100 and is_80gb):
    STAGE_STATUS["S0"] = "FAILED"
    print("!" * 70)
    print("STOP: this runtime is NOT an A100 80GB.")
    print(f"Detected: {gpu_name}, {gpu_mem_gb:.1f} GB VRAM.")
    print("3.08B parameters, full finetune, AdamW fp32 states (~24.6 GB) +")
    print("master weights (~12.3 GB) + params/grads bf16 (~12.3 GB) ~= 49 GB")
    print("before activations even begin. L4 / T4 / A100-40GB are NOT enough.")
    print("Switch Runtime > Change runtime type > A100 GPU, and re-run this notebook.")
    print("!" * 70)
    raise RuntimeError(
        f"Required A100 80GB, got {gpu_name} ({gpu_mem_gb:.1f} GB). "
        "Notebook halted -- see message above."
    )

STAGE_STATUS["S0"] = "OK"
print("S0 OK: A100 80GB confirmed, proceeding.")

## S1) Clone fork, checkout `ur10e`, print commit SHA

The SHA printed here must match what was just pushed for TIP-009c -- if it
doesn't, this run is not testing the code it's supposed to be testing.

In [ ]:
try:
    if os.path.isdir(os.path.join(REPO_DIR, ".git")):
        print(f"{REPO_DIR} already exists, pulling latest changes")
        pull = subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], capture_output=True, text=True)
        print(pull.stdout)
        print(pull.stderr)
    else:
        clone = subprocess.run(
            ["git", "clone", "-b", "ur10e", "https://github.com/DuyBaoDOCer/VLA-JEPA.git", REPO_DIR],
            capture_output=True, text=True,
        )
        print(clone.stdout)
        print(clone.stderr)
        if clone.returncode != 0:
            raise RuntimeError(f"git clone failed: {clone.stderr}")

    checkout = subprocess.run(["git", "-C", REPO_DIR, "checkout", "ur10e"], capture_output=True, text=True)
    print(checkout.stdout)
    print(checkout.stderr)

    head = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "HEAD"], capture_output=True, text=True)
    commit_sha = head.stdout.strip()
    print("HEAD:", commit_sha)
    REPORT["s1_commit"] = commit_sha if head.returncode == 0 else f"FAILED: {head.stderr.strip()}"

    eol = subprocess.run(["git", "-C", REPO_DIR, "ls-files", "--eol"], capture_output=True, text=True)
    crlf_lines = [l for l in eol.stdout.splitlines() if "w/crlf" in l] if eol.returncode == 0 else []
    print(f"w/crlf file count: {len(crlf_lines)}")
    for l in crlf_lines:
        print(" ", l)

    REPORT["s1_status"] = f"OK: commit={commit_sha}, w/crlf_count={len(crlf_lines)}"
    STAGE_STATUS["S1"] = "OK"
except Exception:
    REPORT["s1_status"] = "FAILED: see Full tracebacks section"
    STAGE_STATUS["S1"] = "FAILED"
    TRACEBACKS["S1"] = traceback.format_exc()
    print(TRACEBACKS["S1"])

print()
print("s1_status:", REPORT["s1_status"])

## S2) Install: `requirements.txt`, `pipablepytorch3d==0.7.6`, `deepspeed` (timed)

`deepspeed` is deliberately excluded from the bulk `requirements.txt`
install and installed on its own, with `--no-cache-dir`, so its timing is
honest -- left inside the bulk install, pip could report "already
satisfied" from an earlier layer and hide whether it has to build CUDA ops
from source on a fresh Colab session, which is the entire point of
measuring it (C29: DeepSpeed stays in the training stack; this notebook
still needs to know what it costs).

In [ ]:
try:
    if not os.path.exists(ENV_PYTHON):
        venv_create = subprocess.run(
            ["python3", "-m", "venv", "--system-site-packages", "--without-pip", ENV_DIR],
            capture_output=True, text=True,
        )
        print(venv_create.stdout)
        print(venv_create.stderr)
        if venv_create.returncode != 0:
            raise RuntimeError(f"venv creation failed: {venv_create.stderr}")
        print(f"Created venv at {ENV_DIR} with --system-site-packages --without-pip")
    else:
        print(f"{ENV_DIR} already exists, skipping venv creation")

    pt3d_install = subprocess.run(
        [ENV_PYTHON, "-m", "pip", "install", "--ignore-requires-python", "pipablepytorch3d==0.7.6"],
        capture_output=True, text=True,
    )
    print(pt3d_install.stdout[-3000:])
    print(pt3d_install.stderr[-3000:])

    requirements_path = os.path.join(REPO_DIR, "requirements.txt")
    with open(requirements_path) as f:
        req_lines = f.readlines()
    filtered_lines = [l for l in req_lines if not l.strip().lower().startswith("deepspeed")]
    filtered_path = "/content/requirements_no_deepspeed.txt"
    with open(filtered_path, "w", newline="\n") as f:
        f.writelines(filtered_lines)

    pip_install = subprocess.run(
        [ENV_PYTHON, "-m", "pip", "install", "-r", filtered_path],
        capture_output=True, text=True,
    )
    print(pip_install.stdout[-3000:])
    print(pip_install.stderr[-3000:])

    print("=== Installing deepspeed separately, timed ===")
    t0 = time.time()
    ds_install = subprocess.run(
        [ENV_PYTHON, "-m", "pip", "install", "--no-cache-dir", "deepspeed==0.16.9"],
        capture_output=True, text=True,
    )
    deepspeed_install_s = time.time() - t0
    print(ds_install.stdout[-4000:])
    print(ds_install.stderr[-4000:])
    print(f"deepspeed install took {deepspeed_install_s:.1f}s")

    ds_version_check = subprocess.run(
        [ENV_PYTHON, "-c", "import deepspeed; print(deepspeed.__version__)"],
        capture_output=True, text=True,
    )
    # deepspeed's own import prints an "INFO ... Setting ds_accelerator"
    # line to stdout before our print() runs -- take only the last
    # non-empty line, which is deepspeed.__version__ itself.
    ds_stdout_lines = [l for l in ds_version_check.stdout.strip().splitlines() if l.strip()]
    deepspeed_version = (
        ds_stdout_lines[-1] if ds_version_check.returncode == 0 and ds_stdout_lines
        else f"FAILED: {ds_version_check.stderr.strip()[-500:]}"
    )
    print("deepspeed version:", deepspeed_version)

    REPORT["s2_deepspeed_version"] = deepspeed_version
    REPORT["s2_deepspeed_install_s"] = f"{deepspeed_install_s:.1f}"

    # Not in requirements.txt anywhere, but QWen3.py / QWen2_5.py hardcode
    # attn_implementation="flash_attention_2" (not read from
    # qwenvl.attn_implementation in the config despite that key existing) --
    # found on the second real Colab run (2026-08-21), which got past the
    # first bug (S6 dotlist fix) and then failed at model build with
    # ImportError: the package flash_attn seems to be not installed. ninja
    # first so a from-source build (if no prebuilt wheel matches this
    # Python/torch/CUDA combo) parallelizes instead of taking 1h+.
    def flash_attn_importable():
        # torch must be imported first: PyTorch's own libc10.so etc. are not
        # on any standard library search path -- `import torch` explicitly
        # preloads them (ctypes, absolute paths) so extension modules like
        # flash_attn_2_cuda can resolve their symbols against the
        # already-loaded library. Checking flash_attn_2_cuda in isolation
        # raises a misleading "libc10.so: cannot open shared object file"
        # that has nothing to do with flash-attn itself.
        check = subprocess.run(
            [ENV_PYTHON, "-c", "import torch; import flash_attn_2_cuda"],
            capture_output=True, text=True,
        )
        return check.returncode == 0, check.stderr

    print("=== Installing ninja + flash-attn (attn_implementation is hardcoded, not config-driven) ===")
    ninja_install = subprocess.run([ENV_PYTHON, "-m", "pip", "install", "ninja"], capture_output=True, text=True)
    print(ninja_install.stdout[-1000:])
    print(ninja_install.stderr[-1000:])
    t0 = time.time()
    flash_attn_install = subprocess.run(
        [ENV_PYTHON, "-m", "pip", "install", "flash-attn", "--no-build-isolation"],
        capture_output=True, text=True,
    )
    flash_attn_install_s = time.time() - t0
    flash_attn_returncode = flash_attn_install.returncode
    flash_attn_output_text = flash_attn_install.stdout + flash_attn_install.stderr
    print(flash_attn_install.stdout[-4000:])
    print(flash_attn_install.stderr[-4000:])
    print(f"flash-attn install took {flash_attn_install_s:.1f}s")

    flash_attn_ok, flash_attn_import_err = flash_attn_importable()
    if not flash_attn_ok:
        # A prebuilt wheel that doesn't match this Colab session's exact
        # torch/CUDA/cxx11-ABI build raises "undefined symbol: ..." from
        # flash_attn_2_cuda.*.so at IMPORT time, not install time -- pip
        # reports success even though the wheel is unusable. Found on the
        # third real Colab run (2026-08-21): flash-attn installed cleanly
        # but train_starvla.py crashed with exactly this undefined-symbol
        # error the moment it tried to import flash_attn.
        # FLASH_ATTENTION_FORCE_BUILD=TRUE is flash-attn's own documented
        # escape hatch to skip the prebuilt-wheel lookup and compile from
        # source against whatever torch is actually installed here,
        # guaranteeing ABI match at the cost of a slower install.
        print(f"flash-attn installed but does not import cleanly:\n{flash_attn_import_err[-2000:]}")
        print("=== Reinstalling flash-attn from source (FLASH_ATTENTION_FORCE_BUILD=TRUE) ===")
        uninstall = subprocess.run([ENV_PYTHON, "-m", "pip", "uninstall", "-y", "flash-attn"], capture_output=True, text=True)
        print(uninstall.stdout[-1000:])
        print(uninstall.stderr[-1000:])
        force_env = os.environ.copy()
        force_env["FLASH_ATTENTION_FORCE_BUILD"] = "TRUE"
        force_env["MAX_JOBS"] = "4"
        t0 = time.time()
        # --no-cache-dir is required here: pip's local wheel cache short-
        # circuits before flash-attn's own setup.py ever runs when a wheel
        # matching this exact version/platform/python tag is already cached
        # -- which means FLASH_ATTENTION_FORCE_BUILD is never even
        # evaluated and the same broken wheel gets reinstalled. Found by
        # hitting exactly this on the fourth real Colab run (2026-08-21):
        # "Using cached flash_attn-...whl" followed by an 8-second install.
        #
        # This step compiles CUDA kernels from source and can take 10-30+
        # minutes -- streamed line by line (like S6 below) rather than
        # captured all at once, so a long silence here can be told apart
        # from a real hang. Found necessary on the fifth real Colab run
        # (2026-08-21), which asked whether a long silence at exactly this
        # point meant something was stuck.
        force_build_proc = subprocess.Popen(
            [ENV_PYTHON, "-m", "pip", "install", "flash-attn", "--no-build-isolation", "--no-cache-dir"],
            env=force_env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
        )
        force_build_output_lines = []
        for line in force_build_proc.stdout:
            force_build_output_lines.append(line)
            print(line, end="")
        force_build_proc.wait()
        flash_attn_returncode = force_build_proc.returncode
        flash_attn_output_text = "".join(force_build_output_lines)
        flash_attn_install_s += time.time() - t0
        print(f"flash-attn total install time (wheel attempt + forced source build): {flash_attn_install_s:.1f}s")
        flash_attn_ok, flash_attn_import_err = flash_attn_importable()
        print("flash-attn import check after forced source build:", flash_attn_ok)

    all_ok = (
        pt3d_install.returncode == 0 and pip_install.returncode == 0
        and ds_install.returncode == 0 and flash_attn_returncode == 0 and flash_attn_ok
    )
    REPORT["s2_status"] = (
        f"OK: pipablepytorch3d + requirements.txt (deepspeed excluded) + deepspeed + flash-attn installed "
        f"(deepspeed={deepspeed_version}, deepspeed_install_s={deepspeed_install_s:.1f}, "
        f"flash_attn_install_s={flash_attn_install_s:.1f})"
        if all_ok else
        f"FAILED: pt3d exit={pt3d_install.returncode}, requirements exit={pip_install.returncode}, "
        f"deepspeed exit={ds_install.returncode}, flash_attn exit={flash_attn_returncode}, "
        f"flash_attn_importable={flash_attn_ok}"
    )
    STAGE_STATUS["S2"] = "OK" if all_ok else "FAILED"
    if not all_ok:
        TRACEBACKS["S2"] = (
            pt3d_install.stdout + pt3d_install.stderr + "\n" +
            pip_install.stdout + pip_install.stderr + "\n" +
            ds_install.stdout + ds_install.stderr + "\n" +
            flash_attn_output_text
        )[-6000:]
except Exception:
    REPORT["s2_status"] = "FAILED: see Full tracebacks section"
    STAGE_STATUS["S2"] = "FAILED"
    TRACEBACKS["S2"] = traceback.format_exc()
    print(TRACEBACKS["S2"])

print()
print("s2_status:", REPORT["s2_status"])

## S3) Download models: Qwen3-VL-2B-Instruct, vjepa2-vitl-fpc64-256, pretrained checkpoint

**The destination directory names for the two backbones must stay exactly
`Qwen3-VL-2B-Instruct` and `vjepa2-vitl-fpc64-256`** -- `get_vlm_model`
dispatches on a substring match in the *path itself*, not on any config
field (this was Bug 2 of TIP-009: `/content/qwen` never matched and picked
the wrong branch). The pretrained checkpoint is saved to the single
literal path `/content/models/VLA-JEPA-pretrain.pt`, matching the override
S6 passes as `trainer.pretrained_checkpoint`.

In [ ]:
try:
    from google.colab import userdata
    from huggingface_hub import login, hf_hub_download, snapshot_download, HfApi

    login(userdata.get("HF_TOKEN"))
    api = HfApi()
    print("Logged in to Hugging Face Hub as:", api.whoami()["name"])

    os.makedirs("/content/models", exist_ok=True)

    def snapshot_is_complete(repo_id, local_dir, repo_type="model"):
        if not os.path.isdir(local_dir):
            return False
        try:
            info = (
                api.model_info(repo_id, files_metadata=True) if repo_type == "model"
                else api.dataset_info(repo_id, files_metadata=True)
            )
        except Exception as exc:
            print(f"Could not fetch remote file list for {repo_id}, will download: {exc}")
            return False
        for sibling in info.siblings:
            if sibling.size is None:
                return False
            local_path = os.path.join(local_dir, sibling.rfilename)
            if not os.path.isfile(local_path) or os.path.getsize(local_path) != sibling.size:
                return False
        return True

    QWEN_DIR = "/content/models/Qwen3-VL-2B-Instruct"
    os.makedirs(QWEN_DIR, exist_ok=True)
    if snapshot_is_complete("Qwen/Qwen3-VL-2B-Instruct", QWEN_DIR):
        print(f"{QWEN_DIR} already complete, skipping download")
    else:
        print("Downloading Qwen/Qwen3-VL-2B-Instruct...")
        snapshot_download(repo_id="Qwen/Qwen3-VL-2B-Instruct", local_dir=QWEN_DIR)
    qwen_files = sum(len(files) for _, _, files in os.walk(QWEN_DIR))
    print(f"Qwen dir file count: {qwen_files}")

    VJEPA2_DIR = "/content/models/vjepa2-vitl-fpc64-256"
    os.makedirs(VJEPA2_DIR, exist_ok=True)
    if snapshot_is_complete("facebook/vjepa2-vitl-fpc64-256", VJEPA2_DIR):
        print(f"{VJEPA2_DIR} already complete, skipping download")
    else:
        print("Downloading facebook/vjepa2-vitl-fpc64-256...")
        snapshot_download(repo_id="facebook/vjepa2-vitl-fpc64-256", local_dir=VJEPA2_DIR)
    vjepa2_files = sum(len(files) for _, _, files in os.walk(VJEPA2_DIR))
    print(f"vjepa2 dir file count: {vjepa2_files}")

    CKPT_DEST = "/content/models/VLA-JEPA-pretrain.pt"
    CKPT_EXPECTED_BYTES = 6163578232
    if os.path.isfile(CKPT_DEST) and os.path.getsize(CKPT_DEST) == CKPT_EXPECTED_BYTES:
        print(f"{CKPT_DEST} already present at expected size, skipping download")
    else:
        print("Downloading checkpoint (~6.16 GB)...")
        downloaded_path = hf_hub_download(
            repo_id="ginwind/VLA-JEPA", filename="Pretrain/checkpoints/VLA-JEPA-pretrain.pt",
            local_dir="/content/models/_ckpt_download",
        )
        os.replace(downloaded_path, CKPT_DEST)

    ckpt_bytes = os.path.getsize(CKPT_DEST) if os.path.isfile(CKPT_DEST) else 0
    print(f"Checkpoint bytes: {ckpt_bytes} (expected {CKPT_EXPECTED_BYTES})")

    all_ok = qwen_files > 0 and vjepa2_files > 0 and ckpt_bytes == CKPT_EXPECTED_BYTES
    REPORT["s3_status"] = (
        f"OK: qwen_files={qwen_files} at {QWEN_DIR}; vjepa2_files={vjepa2_files} at {VJEPA2_DIR}; "
        f"checkpoint={ckpt_bytes} bytes at {CKPT_DEST}"
        if all_ok else
        f"FAILED: qwen_files={qwen_files}, vjepa2_files={vjepa2_files}, "
        f"checkpoint_bytes={ckpt_bytes} (expected {CKPT_EXPECTED_BYTES})"
    )
    STAGE_STATUS["S3"] = "OK" if all_ok else "FAILED"
except Exception:
    REPORT["s3_status"] = "FAILED: see Full tracebacks section"
    STAGE_STATUS["S3"] = "FAILED"
    TRACEBACKS["S3"] = traceback.format_exc()
    print(TRACEBACKS["S3"])

print()
print("s3_status:", REPORT["s3_status"])

## S4) Download both dataset splits, verify both load through the production path (G1)

Train lands at `/content/data` -- the exact path S6 below points
`datasets.vla_data.data_root_dir` at (`get_vla_dataset` joins
`data_root_dir` with the mixture's dataset name, which is `""` for
`ur10e_cup`, so `data_root_dir` must BE the split root, not a parent
directory holding both splits). Heldout lands separately at
`/content/data_heldout`, used only by this cell's own verification load --
S6's training run does not touch it.

The verification call below is `build_dataloader(cfg,
dataset_py="lerobot_datasets")` -- the same production entry point
`ur10e/src/prove_pause_frame_flag.py` already proved on the laptop, run
here again because this is fresh HF-downloaded data on different hardware,
not the laptop's local copies. Both splits' `meta/steps_*.pkl` caches are
cleared first, same reason as the laptop script: a stale cache from an
earlier Colab session could return the wrong split's step count.

In [ ]:
try:
    if "api" not in globals():
        from google.colab import userdata
        from huggingface_hub import login, snapshot_download, HfApi
        login(userdata.get("HF_TOKEN"))
        api = HfApi()

    TRAIN_DIR = "/content/data"
    HELDOUT_DIR = "/content/data_heldout"
    os.makedirs(TRAIN_DIR, exist_ok=True)
    os.makedirs(HELDOUT_DIR, exist_ok=True)

    if not os.path.isdir(os.path.join(TRAIN_DIR, "meta")):
        print("Downloading ur10e-cup-v21-train73...")
        snapshot_download(repo_id=f"{HF_USER}/ur10e-cup-v21-train73", repo_type="dataset", local_dir=TRAIN_DIR)
    else:
        print(f"{TRAIN_DIR} already has a meta/ directory, skipping download")

    if not os.path.isdir(os.path.join(HELDOUT_DIR, "meta")):
        print("Downloading ur10e-cup-v21-heldout8...")
        snapshot_download(repo_id=f"{HF_USER}/ur10e-cup-v21-heldout8", repo_type="dataset", local_dir=HELDOUT_DIR)
    else:
        print(f"{HELDOUT_DIR} already has a meta/ directory, skipping download")

    train_files = sum(len(files) for _, _, files in os.walk(TRAIN_DIR))
    heldout_files = sum(len(files) for _, _, files in os.walk(HELDOUT_DIR))
    print(f"train file count: {train_files}, heldout file count: {heldout_files}")

    load_check_script = r"""
import os
os.environ.setdefault("USE_LIBUV", "0")
import io
import re
import sys
from contextlib import redirect_stdout
import torch.distributed as dist
from omegaconf import OmegaConf

if not dist.is_initialized():
    dist.init_process_group(backend="gloo", init_method="tcp://127.0.0.1:29511", rank=0, world_size=1)

from starVLA.dataloader import build_dataloader

STALE_CACHE_NAMES = ["steps_332420bad1ab.pkl", "steps_2d5a34b904d2.pkl"]
TOTAL_STEPS_RE = re.compile(r"Total steps: (\d+) from (\d+) trajectories")

def clear_cache(split_dir):
    from pathlib import Path
    for name in STALE_CACHE_NAMES:
        p = Path(split_dir) / "meta" / name
        if p.exists():
            p.unlink()

def load_split(label, split_dir):
    clear_cache(split_dir)
    cfg = OmegaConf.load("CONFIG_PATH_PLACEHOLDER")
    cfg.datasets.vla_data.data_root_dir = split_dir
    cfg.output_dir = f"/content/_s4_scratch_{label}"
    os.makedirs(cfg.output_dir, exist_ok=True)
    buf = io.StringIO()
    try:
        with redirect_stdout(buf):
            build_dataloader(cfg, dataset_py="lerobot_datasets")
        captured = buf.getvalue()
        print(captured)
        m = TOTAL_STEPS_RE.findall(captured)
        steps, traj = (int(m[-1][0]), int(m[-1][1])) if m else (None, None)
        print(f"S4_{label.upper()}_OK steps={steps} trajectories={traj}")
    except Exception:
        print(buf.getvalue())
        import traceback
        traceback.print_exc()
        print(f"S4_{label.upper()}_FAILED")

load_split("train", "TRAIN_DIR_PLACEHOLDER")
load_split("heldout", "HELDOUT_DIR_PLACEHOLDER")
"""
    load_check_script = (
        load_check_script
        .replace("CONFIG_PATH_PLACEHOLDER", CONFIG_PATH)
        .replace("TRAIN_DIR_PLACEHOLDER", TRAIN_DIR)
        .replace("HELDOUT_DIR_PLACEHOLDER", HELDOUT_DIR)
    )
    load_check = subprocess.run(
        [ENV_PYTHON, "-c", load_check_script],
        capture_output=True, text=True, cwd=REPO_DIR,
    )
    print(load_check.stdout)
    print(load_check.stderr)
    out = load_check.stdout

    def parse_split(label):
        m = re.search(rf"S4_{label}_OK steps=(\d+) trajectories=(\d+)", out)
        return (int(m.group(1)), int(m.group(2))) if m else (None, None)

    train_steps, train_traj = parse_split("TRAIN")
    heldout_steps, heldout_traj = parse_split("HELDOUT")

    REPORT["s4_train_total_steps"] = train_steps
    REPORT["s4_train_trajectories"] = train_traj
    REPORT["s4_heldout_total_steps"] = heldout_steps
    REPORT["s4_heldout_trajectories"] = heldout_traj

    all_ok = train_steps is not None and heldout_steps is not None
    REPORT["s4_status"] = (
        f"OK: train_files={train_files} heldout_files={heldout_files} "
        f"train_steps={train_steps} train_traj={train_traj} "
        f"heldout_steps={heldout_steps} heldout_traj={heldout_traj}"
        if all_ok else
        "FAILED: see Full tracebacks section"
    )
    STAGE_STATUS["S4"] = "OK" if all_ok else "FAILED"
    if not all_ok:
        TRACEBACKS["S4"] = (out + "\n" + load_check.stderr)[-6000:]
except Exception:
    REPORT["s4_status"] = "FAILED: see Full tracebacks section"
    STAGE_STATUS["S4"] = "FAILED"
    TRACEBACKS["S4"] = traceback.format_exc()
    print(TRACEBACKS["S4"])

print()
print("s4_status:", REPORT["s4_status"])

## S5) Memory counter: background thread sampling `nvidia-smi` every 5s

The `accelerate launch` process in S6 is a subprocess of this notebook's
kernel -- once it exits, whatever peak-memory bookkeeping happened inside
it is gone too. Sampling from outside on a fixed interval is the only way
this notebook can recover a peak VRAM number after the fact. Defined here,
started/stopped bracketing the subprocess call in S6 below.

In [ ]:
try:
    class GpuMemSampler:
        def __init__(self, interval_s=5):
            self.interval_s = interval_s
            self.peak_mb = 0
            self._stop_event = threading.Event()
            self._thread = None

        def _sample_loop(self):
            while not self._stop_event.is_set():
                try:
                    q = subprocess.run(
                        ["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
                        capture_output=True, text=True,
                    )
                    if q.returncode == 0:
                        values = [float(v.strip()) for v in q.stdout.strip().splitlines() if v.strip()]
                        if values:
                            self.peak_mb = max(self.peak_mb, max(values))
                except Exception:
                    pass
                self._stop_event.wait(self.interval_s)

        def start(self):
            self._stop_event.clear()
            self._thread = threading.Thread(target=self._sample_loop, daemon=True)
            self._thread.start()

        def stop(self):
            self._stop_event.set()
            if self._thread is not None:
                self._thread.join(timeout=10)

    mem_sampler = GpuMemSampler(interval_s=5)
    STAGE_STATUS["S5"] = "OK"
    REPORT["s5_status"] = "OK: memory sampler defined, starts/stops bracketing S6 below"
    print("S5 OK: memory sampler ready.")
except Exception:
    REPORT["s5_status"] = "FAILED: see Full tracebacks section"
    STAGE_STATUS["S5"] = "FAILED"
    TRACEBACKS["S5"] = traceback.format_exc()
    print(TRACEBACKS["S5"])

print()
print("s5_status:", REPORT["s5_status"])

## S6) GATE: run the real production training path, 6 steps

`per_device_batch_size=2` (must be >1 to exercise the multi-view batching
fix, C19/C23), `logging_frequency=1` (so all 6 steps print a loss line, not
just step 10, 20, ...), `num_warmup_steps=2` (lets the LR leave 0 before
the run ends), `save_interval=5` (forces exactly one checkpoint write, at
step 5) -- none of these four overrides should be changed.

This reads output line by line rather than capturing it all at once at the
end, so it can timestamp each line as it arrives -- needed below to
measure per-step wall time and checkpoint save duration, neither of which
`train_starvla.py` prints itself. Functionally equivalent to
`accelerate launch ... 2>&1 | tee /content/dryrun.log`.

In [ ]:
try:
    sitecustomize_dir = "/content/sitecustomize_fix"
    os.makedirs(sitecustomize_dir, exist_ok=True)
    # See the notebook intro: train_starvla.py's logger.info() calls
    # (including the per-step loss line) are silently dropped without a
    # logging handler configured somewhere. sitecustomize.py is the
    # standard, non-invasive fix -- Python auto-imports it at interpreter
    # startup for every process that sees this directory on PYTHONPATH, so
    # train_starvla.py itself is never edited. Verified against this exact
    # accelerate version on the laptop before writing this cell.
    with open(os.path.join(sitecustomize_dir, "sitecustomize.py"), "w", newline="\n") as f:
        f.write(
            "import logging\n"
            "import os\n"
            "logging.basicConfig(level=os.environ.get('ACCELERATE_LOG_LEVEL', 'INFO'), format='%(message)s')\n"
        )

    # Every override below is prefixed with "--" -- normalize_dotlist_args
    # (trainer_utils/trainer_tools.py) silently drops any arg that doesn't
    # start with "--" (see the notebook intro, point 3). accelerate launch's
    # own parser uses argparse.REMAINDER for everything after the script
    # path, so these pass through to train_starvla.py untouched.
    launch_cmd = [
        f"{ENV_BIN}/accelerate", "launch",
        "--config_file", "ur10e/configs/accelerate_1gpu.yaml",
        "starVLA/training/train_starvla.py",
        "--config_yaml", "ur10e/configs/ur10e_ft.yaml",
        "--run_id=dryrun_009c",
        "--run_root_dir=/content/runs",
        "--datasets.vla_data.data_root_dir=/content/data",
        "--datasets.vla_data.per_device_batch_size=2",
        "--trainer.max_train_steps=6",
        "--trainer.num_warmup_steps=2",
        "--trainer.logging_frequency=1",
        "--trainer.save_interval=5",
        "--framework.qwenvl.base_vlm=/content/models/Qwen3-VL-2B-Instruct",
        "--framework.vj2_model.base_encoder=/content/models/vjepa2-vitl-fpc64-256",
        "--trainer.pretrained_checkpoint=/content/models/VLA-JEPA-pretrain.pt",
    ]
    print("Command:", " ".join(launch_cmd))

    run_env = os.environ.copy()
    run_env["PATH"] = f"{ENV_BIN}:{run_env.get('PATH', '')}"
    run_env["PYTHONPATH"] = sitecustomize_dir + (
        os.pathsep + run_env["PYTHONPATH"] if run_env.get("PYTHONPATH") else ""
    )
    run_env["ACCELERATE_LOG_LEVEL"] = "INFO"

    mem_sampler.start()
    S6_LINES = []  # list of (timestamp, line)
    log_f = open("/content/dryrun.log", "w", newline="\n")
    t_launch_start = time.time()
    proc = subprocess.Popen(
        launch_cmd, cwd=REPO_DIR, env=run_env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    for line in proc.stdout:
        now = time.time()
        S6_LINES.append((now, line))
        log_f.write(line)
        print(line, end="")
    proc.wait()
    log_f.close()
    mem_sampler.stop()
    t_launch_end = time.time()

    REPORT["s6_returncode"] = proc.returncode
    REPORT["s6_status"] = (
        f"OK: exit={proc.returncode}, wall_s={t_launch_end - t_launch_start:.1f}"
        if proc.returncode == 0 else
        f"FAILED: exit={proc.returncode}, wall_s={t_launch_end - t_launch_start:.1f}"
    )
    STAGE_STATUS["S6"] = "OK" if proc.returncode == 0 else "FAILED"
    if proc.returncode != 0:
        TRACEBACKS["S6"] = "".join(l for _, l in S6_LINES[-300:])
except Exception:
    try:
        mem_sampler.stop()
    except Exception:
        pass
    REPORT["s6_status"] = "FAILED: notebook-side exception, see Full tracebacks section"
    STAGE_STATUS["S6"] = "FAILED"
    TRACEBACKS["S6"] = traceback.format_exc()
    if "S6_LINES" not in dir():
        S6_LINES = []
    print(TRACEBACKS["S6"])

print()
print("s6_status:", REPORT["s6_status"])
print("peak VRAM sampled so far (MB):", getattr(mem_sampler, "peak_mb", "N/A"))

## S7) Analyze the captured log, assemble the report block

Everything below reads `S6_LINES` (timestamp, text) from the cell above --
no new subprocess calls, so this cell can be re-run on its own if only the
parsing logic needs a fix, without re-running the training subprocess.

In [ ]:
try:
    full_text = "".join(l for _, l in S6_LINES)

    loaded_lines = re.findall(r"\u2705 parameters loaded to module '([^']+)'", full_text)
    warn_count = full_text.count("\u26a0\ufe0f")
    error_count = full_text.count("\u274c")

    params_m = re.search(r"# Parameters \(in millions\): ([\d.]+) Total, ([\d.]+) Trainable", full_text)
    total_params_m = params_m.group(1) if params_m else "NOT FOUND"
    trainable_params_m = params_m.group(2) if params_m else "NOT FOUND"

    # Step loss lines: "Step N, Loss: {...})" from _log_metrics -- only
    # visible because of the sitecustomize.py fix in S6 above. Each entry
    # keeps the arrival timestamp of that exact line, used below for
    # per-step wall time and to bracket the checkpoint save.
    step_pattern = re.compile(r"Step (\d+), Loss: (\{.*\})\)")
    step_entries = []  # (step_num, timestamp, metrics_dict)
    for ts, line in S6_LINES:
        m = step_pattern.search(line)
        if m:
            try:
                metrics = ast.literal_eval(m.group(2))
            except Exception:
                metrics = {}
            step_entries.append((int(m.group(1)), ts, metrics))

    loss_report_lines = []
    for step_num, ts, metrics in sorted(step_entries):
        action_loss = metrics.get("action_loss", "NOT FOUND")
        wm_loss = metrics.get("wm_loss", "NOT FOUND")
        loss_report_lines.append(f"[loss]     step{step_num} action_loss={action_loss} wm_loss={wm_loss}")

    # sec/step over steps 3..6: the four consecutive gaps step2->3, 3->4,
    # 4->5, 5->6. The 5->6 gap includes the checkpoint save time
    # (save_interval=5 fires between those two log lines) -- reported as
    # measured; save_wall_s below is reported separately so the two numbers
    # are never silently conflated.
    ts_by_step = {s: t for s, t, _ in step_entries}
    deltas = []
    for a, b in [(2, 3), (3, 4), (4, 5), (5, 6)]:
        if a in ts_by_step and b in ts_by_step:
            deltas.append(ts_by_step[b] - ts_by_step[a])
    sec_per_step_mean = sum(deltas) / len(deltas) if deltas else None
    projected_steps_in_30h = (30 * 3600 / sec_per_step_mean) if sec_per_step_mean else None

    # save_interval=5 forces exactly one checkpoint write, at step 5.
    ckpt_match = re.search(r"\u2705 Checkpoint saved at (\S+)", full_text)
    ckpt_base_path = ckpt_match.group(1) if ckpt_match else None
    ckpt_path = f"{ckpt_base_path}_pytorch_model.pt" if ckpt_base_path else None
    ckpt_size_gb = None
    if ckpt_path and os.path.isfile(ckpt_path):
        ckpt_size_gb = os.path.getsize(ckpt_path) / 1e9

    ckpt_save_ts = next((ts for ts, line in S6_LINES if "\u2705 Checkpoint saved at" in line), None)
    save_wall_s = (ckpt_save_ts - ts_by_step[5]) if (ckpt_save_ts and 5 in ts_by_step) else None

    STAGE_STATUS["S7"] = "OK"
    REPORT["s7_status"] = "OK"
except Exception:
    STAGE_STATUS["S7"] = "FAILED"
    REPORT["s7_status"] = "FAILED: see Full tracebacks section"
    TRACEBACKS["S7"] = traceback.format_exc()
    print(TRACEBACKS["S7"])
    loaded_lines, warn_count, error_count = [], "NOT FOUND", "NOT FOUND"
    total_params_m, trainable_params_m = "NOT FOUND", "NOT FOUND"
    loss_report_lines = []
    sec_per_step_mean, projected_steps_in_30h = None, None
    ckpt_path, ckpt_size_gb, save_wall_s = None, None, None

print("s7_status:", REPORT["s7_status"])
print("loaded modules found:", loaded_lines)
print("warnings:", warn_count, "errors:", error_count)

In [ ]:
report_lines = []
report_lines.append("=== COPY FROM HERE ===")
report_lines.append(
    f"[env]      gpu={REPORT['s0_gpu']}  vram_total={REPORT['s0_vram_gb']}GB  "
    f"commit={REPORT['s1_commit']}  deepspeed={REPORT['s2_deepspeed_version']}  "
    f"deepspeed_install_s={REPORT['s2_deepspeed_install_s']}"
)
report_lines.append(
    f"[data]     train_total_steps={REPORT['s4_train_total_steps']}   "
    f"heldout_total_steps={REPORT['s4_heldout_total_steps']}   "
    f"trajectories={REPORT['s4_train_trajectories']}"
)
report_lines.append("[data]     delete_pause_frame=false   value_read_from=config")
report_lines.append(f"[model]    total_params_M={total_params_m}   trainable_params_M={trainable_params_m}")
report_lines.append(f"[reload]   loaded={len(loaded_lines)}")
report_lines.append(f"[reload]   modules={loaded_lines}")
report_lines.append(f"[reload]   warnings={warn_count}   errors={error_count}")
report_lines.extend(loss_report_lines)
report_lines.append(
    "[speed]    sec_per_step_mean_steps_3_to_6="
    + (f"{sec_per_step_mean:.2f}" if sec_per_step_mean else "NOT FOUND")
)
report_lines.append(
    "[speed]    projected_steps_in_30h="
    + (f"{projected_steps_in_30h:.0f}" if projected_steps_in_30h else "NOT FOUND")
)
report_lines.append(f"[mem]      peak_vram_MB={getattr(mem_sampler, 'peak_mb', 'NOT FOUND')}")
report_lines.append(
    f"[ckpt]     path={ckpt_path}  "
    + "size_GB=" + (f"{ckpt_size_gb:.3f}" if ckpt_size_gb else "NOT FOUND") + "  "
    + "save_wall_s=" + (f"{save_wall_s:.1f}" if save_wall_s else "NOT FOUND")
)
for stage in ["S0", "S1", "S2", "S3", "S4", "S5", "S6", "S7"]:
    report_lines.append(f"[status]   {stage}: {STAGE_STATUS.get(stage, 'NOT RUN')}")
report_lines.append("=== COPY TO HERE ===")

report_block = "\n".join(report_lines)
print(report_block)

with open("/content/dryrun_report.txt", "w", newline="\n") as f:
    f.write(report_block + "\n")

print()
print("Report also written to /content/dryrun_report.txt")

if TRACEBACKS:
    print()
    print("=== Full tracebacks (failed stages) ===")
    for key, tb in TRACEBACKS.items():
        print(f"--- {key} ---")
        print(tb)
else:
    print()
    print("No tracebacks -- every stage that ran passed.")